In [0]:
%run ../lib-spark

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


In [0]:
!ip a

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0@if2: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 8881 qdisc noqueue state UP group default 
    link/ether d6:88:cd:2b:54:2f brd ff:ff:ff:ff:ff:ff link-netnsid 0
    inet 192.168.210.33/32 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fddb:face:1::c0a8:d221/128 scope global nodad 
       valid_lft forever preferred_lft forever
    inet6 fe80::d488:cdff:fe2b:542f/64 scope link 
       valid_lft forever preferred_lft forever


In [0]:
%%bash
ssh -o StrictHostKeyChecking=accept-new -o \
    UserKnownHostsFile=/tmp/known_hosts -o BatchMode=yes \
    -p 8889 \
    -i /Workspace/Users/rogermm@gmail.com/.ssh/id_ed25519_databricks_free \
    databricks@pub.worldb.dedyn.io \
    whoami



databricks


In [0]:
#control_path = "/tmp/ssh-databricks-%C"
control_path = "/tmp/ssh-databricks"

ssh_command = (
    "ssh "
    "-o StrictHostKeyChecking=accept-new "
    "-o UserKnownHostsFile=/tmp/known_hosts "
    "-o BatchMode=yes "
    "-o ConnectTimeout=5 "
    "-o ConnectionAttempts=1 "
    "-o ControlMaster=auto "
    f"-o ControlPath={control_path} "
    "-o ControlPersist=10m "
    "-p 8889 "
    "-i /Workspace/Users/rogermm@gmail.com/.ssh/id_ed25519_databricks_free "
    "databricks@pub.worldb.dedyn.io"
)

In [0]:
!{ssh_command} whoami

databricks


In [0]:
!ps -ef | grep "/tmp/ssh-databricks"

spark-7+     329       1  0 19:52 ?        00:00:00 ssh: /tmp/ssh-databricks [mux]
spark-7+     614      85 99 19:59 pts/0    00:00:00 /bin/bash -c ps -ef | grep "/tmp/ssh-databricks"
spark-7+     644     614  0 19:59 pts/0    00:00:00 grep /tmp/ssh-databricks


In [0]:
commands = [
    f"{ssh_command} whoami"
]

result_df = run_worker_commands(commands)

display(result_df)

command,exit_code,output
ssh -o StrictHostKeyChecking=accept-new -o UserKnownHostsFile=/tmp/known_hosts -o BatchMode=yes -p 8889 -i /Workspace/Users/rogermm@gmail.com/.ssh/id_ed25519_databricks_free databricks@pub.worldb.dedyn.io whoami,0,databricks


In [0]:
!ls /Workspace/Users/rogermm@gmail.com
!ls /databricks

Drafts			      old   test-spark-worker-udf-cli-2.ipynb
monorepo-datahub-ops-private  test
airbyte		    jars		   repl-logs
BUILD.bazel	    jdkpatch		   rngtracing
BUILDINFO	    jfr_configs		   runtime
chauffeur	    jvm-startup		   safespark
chauffeur-jars	    kernel-connections	   sandbox-common
common		    keys		   scala-kernel-jars
conda		    licenses		   scala-kernel-scripts
data		    logs		   scripts
dbr-secrets	    miniconda		   secrets
DBR_VERSION	    native-utils	   snapstart
driver		    native-utils.runfiles  spark
executor	    notebook-grpc	   spark-empty-dir
git		    python		   structured_log_parser
hadoop-safety-jars  python3		   ttyd
hive		    python-bootstrap	   usdt
IMAGE_KEY	    python-lsp
init_logs	    python_shell


In [0]:
import subprocess

#CONTROL_SOCKET = "/tmp/databricks-ssh-%C"
CONTROL_SOCKET = "/tmp/databricks-ssh"
SSH_HOST = "databricks@pub.worldb.dedyn.io"
SSH_PORT = 8889
SSH_KEY = "/Workspace/Users/rogermm@gmail.com/.ssh/id_ed25519_databricks_free"

ssh_base = (
    f"ssh "
    f"-S {CONTROL_SOCKET} "
    f"-p {SSH_PORT} "
    f"-i {SSH_KEY} "
    f"-o BatchMode=yes "
    f"{SSH_HOST}"
)


def start_ssh_tunnel() -> None:
    command = (
        f"ssh "
        f"-M "
        f"-S {CONTROL_SOCKET} "
        f"-o ControlPersist=yes "
        f"-o ServerAliveInterval=30 "
        f"-o ServerAliveCountMax=3 "
        f"-o ExitOnForwardFailure=yes "
        f"-o StrictHostKeyChecking=accept-new "
        f"-o UserKnownHostsFile=/tmp/known_hosts "
        f"-o BatchMode=yes "
        f"-p {SSH_PORT} "
        f"-i {SSH_KEY} "
        f"-L 127.0.0.1:9092:kafka-4:9092 "
        f"-fNT "
        f"{SSH_HOST}"
    )

    subprocess.run(command, shell=True, check=True)


def check_ssh_tunnel() -> bool:
    result = subprocess.run(
        f"{ssh_base} -O check",
        shell=True,
        text=True,
        capture_output=True,
    )

    print(result.stdout or result.stderr)
    return result.returncode == 0


def stop_ssh_tunnel() -> None:
    subprocess.run(
        f"{ssh_base} -O exit",
        shell=True,
        check=False,
    )

In [0]:
start_ssh_tunnel()

In [0]:
check_ssh_tunnel()

Master running (pid=697)



True

In [0]:
stop_ssh_tunnel()

Exit request sent.
